# Kuhn Poker Deep CFR Multi-Seed Validation Experiment

This notebook runs one fixed-budget Deep CFR training phase across ten random seeds. The primary validation metric is exploitability, with policy-value error and neural training diagnostics reported as secondary evidence.

In [ ]:
import collections
import csv
import json
import math
import os
import random
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from scipy import stats
from torch import nn
from tqdm import tqdm

import pyspiel
from open_spiel.python import policy
from open_spiel.python.algorithms import expected_game_score
from open_spiel.python.algorithms import exploitability

# Keep execution CPU-only for reproducibility unless you intentionally enable GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("Torch CUDA version:", torch.version.cuda)


In [ ]:
AdvantageMemory = collections.namedtuple(
    "AdvantageMemory", "info_state iteration advantage action")

StrategyMemory = collections.namedtuple(
    "StrategyMemory", "info_state iteration strategy_action_probs")


In [ ]:
class SonnetLinear(nn.Module):
  """A Sonnet linear module.

  Always includes biases and only supports ReLU activations.
  """

  def __init__(self, in_size, out_size, activate_relu=True):
    """Creates a Sonnet linear layer.

    Args:
      in_size: (int) number of inputs
      out_size: (int) number of outputs
      activate_relu: (bool) whether to include a ReLU activation layer
    """
    super(SonnetLinear, self).__init__()
    self._activate_relu = activate_relu
    self._in_size = in_size
    self._out_size = out_size
    # stddev = 1.0 / math.sqrt(self._in_size)
    # mean = 0
    # lower = (-2 * stddev - mean) / stddev
    # upper = (2 * stddev - mean) / stddev
    # # Weight initialization inspired by Sonnet's Linear layer,
    # # which cites https://arxiv.org/abs/1502.03167v3
    # # pytorch default: initialized from
    # # uniform(-sqrt(1/in_features), sqrt(1/in_features))
    self._weight = None
    self._bias = None
    self.reset()

  def forward(self, tensor):
    y = F.linear(tensor, self._weight, self._bias)
    return F.relu(y) if self._activate_relu else y

  def reset(self):
    stddev = 1.0 / math.sqrt(self._in_size)
    mean = 0
    lower = (-2 * stddev - mean) / stddev
    upper = (2 * stddev - mean) / stddev
    # Weight initialization inspired by Sonnet's Linear layer,
    # which cites https://arxiv.org/abs/1502.03167v3
    # pytorch default: initialized from
    # uniform(-sqrt(1/in_features), sqrt(1/in_features))
    self._weight = nn.Parameter(
        torch.Tensor(
            stats.truncnorm.rvs(
                lower,
                upper,
                loc=mean,
                scale=stddev,
                size=[self._out_size, self._in_size])))
    self._bias = nn.Parameter(torch.zeros([self._out_size]))

In [ ]:
class MLP(nn.Module):
  """A simple network built from nn.linear layers."""

  def __init__(self,
               input_size,
               hidden_sizes,
               output_size,
               activate_final=False):
    """Create the MLP.

    Args:
      input_size: (int) number of inputs
      hidden_sizes: (list) sizes (number of units) of each hidden layer
      output_size: (int) number of outputs
      activate_final: (bool) should final layer should include a ReLU
    """

    super(MLP, self).__init__()
    self._layers = []
    # Hidden layers
    for size in hidden_sizes:
      self._layers.append(SonnetLinear(in_size=input_size, out_size=size))
      input_size = size
    # Output layer
    self._layers.append(
        SonnetLinear(
            in_size=input_size,
            out_size=output_size,
            activate_relu=activate_final))

    self.model = nn.ModuleList(self._layers)

  def forward(self, x):
    for layer in self.model:
      x = layer(x)
    return x

  def reset(self):
    for layer in self._layers:
      layer.reset()

In [ ]:
class ReservoirBuffer(object):
  """Allows uniform sampling over a stream of data.

  This class supports the storage of arbitrary elements, such as observation
  tensors, integer actions, etc.
  See https://en.wikipedia.org/wiki/Reservoir_sampling for more details.
  """

  def __init__(self, reservoir_buffer_capacity):
    self._reservoir_buffer_capacity = reservoir_buffer_capacity
    self._data = []
    self._add_calls = 0

  def add(self, element):
    """Potentially adds `element` to the reservoir buffer.

    Args:
      element: data to be added to the reservoir buffer.
    """
    if len(self._data) < self._reservoir_buffer_capacity:
      self._data.append(element)
    else:
      idx = np.random.randint(0, self._add_calls + 1)
      if idx < self._reservoir_buffer_capacity:
        self._data[idx] = element
    self._add_calls += 1

  def sample(self, num_samples):
    """Returns `num_samples` uniformly sampled from the buffer.

    Args:
      num_samples: `int`, number of samples to draw.

    Returns:
      An iterable over `num_samples` random elements of the buffer.
    Raises:
      ValueError: If there are less than `num_samples` elements in the buffer
    """
    if len(self._data) < num_samples:
      raise ValueError("{} elements could not be sampled from size {}".format(
          num_samples, len(self._data)))
    return random.sample(self._data, num_samples)

  def clear(self):
    self._data = []
    self._add_calls = 0

  def __len__(self):
    return len(self._data)

  def __iter__(self):
    return iter(self._data)

In [ ]:
class DeepCFRSolver(policy.Policy):
  """Implements a solver for the Deep CFR Algorithm with PyTorch.

  See https://arxiv.org/abs/1811.00164.

  Define all networks and sampling buffers/memories.  Derive losses & learning
  steps. Initialize the game state and algorithmic variables.

  Note: batch sizes default to `None` implying that training over the full
        dataset in memory is done by default.  To sample from the memories you
        may set these values to something less than the full capacity of the
        memory.
  """

  def __init__(self,
               game,
               policy_network_layers=(256, 256),
               advantage_network_layers=(128, 128),
               num_iterations: int = 100,
               num_traversals: int = 20,
               learning_rate: float = 1e-4,
               batch_size_advantage=None,
               batch_size_strategy=None,
               memory_capacity: int = int(1e6),
               policy_network_train_steps: int = 1,
               policy_network_train_every: int = 1,   # NEW
               advantage_network_train_steps: int = 1,
               reinitialize_advantage_networks: bool = True,
               compute_exploitability: bool = False,           
               ):
    """Initialize the Deep CFR algorithm.

    Args:
      game: Open Spiel game.
      policy_network_layers: (list[int]) Layer sizes of strategy net MLP.
      advantage_network_layers: (list[int]) Layer sizes of advantage net MLP.
      num_iterations: (int) Number of training iterations.
      num_traversals: (int) Number of traversals per iteration.
      learning_rate: (float) Learning rate.
      batch_size_advantage: (int or None) Batch size to sample from advantage
        memories.
      batch_size_strategy: (int or None) Batch size to sample from strategy
        memories.
      memory_capacity: Number af samples that can be stored in memory.
      policy_network_train_steps: Number of policy network training steps (per
        iteration).
      advantage_network_train_steps: Number of advantage network training steps
        (per iteration).
      reinitialize_advantage_networks: Whether to re-initialize the advantage
        network before training on each iteration.
    """
    all_players = list(range(game.num_players()))
    super(DeepCFRSolver, self).__init__(game, all_players)
    self._game = game
    if game.get_type().dynamics == pyspiel.GameType.Dynamics.SIMULTANEOUS:
      # `_traverse_game_tree` does not take into account this option.
      raise ValueError("Simulatenous games are not supported.")
    self._batch_size_advantage = batch_size_advantage
    self._batch_size_strategy = batch_size_strategy
    self._policy_network_train_steps = policy_network_train_steps
    self._advantage_network_train_steps = advantage_network_train_steps
    self._num_players = game.num_players()
    self._root_node = self._game.new_initial_state()
    self._embedding_size = len(self._root_node.information_state_tensor(0))
    self._num_iterations = num_iterations
    self._num_traversals = num_traversals
    self._reinitialize_advantage_networks = reinitialize_advantage_networks
    self._num_actions = game.num_distinct_actions()
    self._iteration = 1

    # Define strategy network, loss & memory.
    self._strategy_memories = ReservoirBuffer(memory_capacity)
    self._policy_network = MLP(self._embedding_size,
                               list(policy_network_layers),
                               self._num_actions)
    # Illegal actions are handled in the traversal code where expected payoff
    # and sampled regret is computed from the advantage networks.
    self._policy_sm = nn.Softmax(dim=-1)
    self._loss_policy = nn.MSELoss()
    self._optimizer_policy = torch.optim.Adam(
        self._policy_network.parameters(), lr=learning_rate)

    # Define advantage network, loss & memory. (One per player)
    self._advantage_memories = [
        ReservoirBuffer(memory_capacity) for _ in range(self._num_players)
    ]
    self._advantage_networks = [
        MLP(self._embedding_size, list(advantage_network_layers),
            self._num_actions) for _ in range(self._num_players)
    ]
    self._loss_advantages = nn.MSELoss(reduction="mean")
    self._optimizer_advantages = []
    for p in range(self._num_players):
      self._optimizer_advantages.append(
          torch.optim.Adam(
              self._advantage_networks[p].parameters(), lr=learning_rate))
    self._learning_rate = learning_rate
    self._compute_exploitability = bool(compute_exploitability)
    self._policy_network_train_every = int(policy_network_train_every)
    if self._policy_network_train_every < 1:
      raise ValueError("policy_network_train_every must be >= 1")

    # Track total game-tree nodes touched during traversal and checkpoint metrics.
    self._nodes_touched = 0
    self._nodes_touched_history = []
    self._average_policy_value_history = []
    self._last_advantage_grad_norm = [np.nan for _ in range(self._num_players)]
    self._last_policy_grad_norm = np.nan



  @property
  def advantage_buffers(self):
    return self._advantage_memories

  @property
  def strategy_buffer(self):
    return self._strategy_memories

  def clear_advantage_buffers(self):
    for p in range(self._num_players):
      self._advantage_memories[p].clear()

  def reinitialize_advantage_network(self, player):
    self._advantage_networks[player].reset()
    self._optimizer_advantages[player] = torch.optim.Adam(
        self._advantage_networks[player].parameters(), lr=self._learning_rate)

  def reinitialize_advantage_networks(self):
    for p in range(self._num_players):
      self.reinitialize_advantage_network(p)


  @staticmethod
  def _as_scalar_iteration(value):
    """Returns a scalar float iteration weight from a stored replay-buffer value."""
    arr = np.asarray(value)
    if arr.ndim == 0:
      return float(arr)
    flat = arr.reshape(-1)
    if flat.size != 1:
      raise ValueError(
          f"Expected scalar iteration value, got shape {arr.shape} with type {type(value)}")
    return float(flat[0])

  # def solve(self):
  #   """Solution logic for Deep CFR.

  #   Traverses the game tree, while storing the transitions for training
  #   advantage and policy networks.

  #   Returns:
  #     1. (nn.Module) Instance of the trained policy network for inference.
  #     2. (list of floats) Advantage network losses for
  #       each player during each iteration.
  #     3. (float) Policy loss.
  #   """
  #   advantage_losses = collections.defaultdict(list)
  #   for _ in range(self._num_iterations):
  #     for p in range(self._num_players):
  #       for _ in range(self._num_traversals):
  #         self._traverse_game_tree(self._root_node, p)
  #       if self._reinitialize_advantage_networks:
  #         # Re-initialize advantage network for player and train from scratch.
  #         self.reinitialize_advantage_network(p)
  #       # Re-initialize advantage networks and train from scratch.
  #       advantage_losses[p].append(self._learn_advantage_network(p))
  #     self._iteration += 1
  #     # Train policy network.
  #   policy_loss = self._learn_strategy_network()
  #   return self._policy_network, advantage_losses, policy_loss
  
  def solve(self):
    """Runs one fixed-budget Deep CFR training phase.

    Returns:
      1. (nn.Module) Trained policy network.
      2. (dict) Advantage losses per player per iteration.
      3. (list) Policy losses per training checkpoint.
      4. (list) NashConv values at policy checkpoints.
      5. (list) Total nodes touched at each policy checkpoint.
      6. (list) Average-policy values at each policy checkpoint.
      7. (dict) Additional diagnostics recorded at policy checkpoints.
    """
    start_time = time.perf_counter()
    advantage_losses = collections.defaultdict(list)
    policy_losses_at_checkpoints = []
    convs = []
    nodes_touched_history = []
    average_policy_values = []
    diagnostics = collections.defaultdict(list)

    for it in range(self._num_iterations):
      # ---- collect samples and train advantage networks ----
      for p in range(self._num_players):
        for _ in range(self._num_traversals):
          self._traverse_game_tree(self._root_node, p)

        if self._reinitialize_advantage_networks:
          self.reinitialize_advantage_network(p)

        advantage_losses[p].append(self._learn_advantage_network(p))

      # ---- end-of-iteration bookkeeping ----
      self._iteration += 1

      # ---- train and evaluate the average-policy network every K iterations ----
      train_now = (((it + 1) % self._policy_network_train_every) == 0) or (
          it == self._num_iterations - 1)

      if train_now:
        pol_loss = self._learn_strategy_network()
        policy_losses_at_checkpoints.append(pol_loss)

        tab_pol = policy.tabular_policy_from_callable(
            self._game, self.action_probabilities)
        nodes_touched_history.append(self._nodes_touched)

        avg_policy_value = expected_game_score.policy_value(
            self._game.new_initial_state(), [tab_pol] * self._num_players)[0]
        average_policy_values.append(avg_policy_value)

        if self._compute_exploitability:
          conv = exploitability.nash_conv(self._game, tab_pol)
          convs.append(conv)
        else:
          convs.append(np.nan)

        policy_diag = self._policy_network_diagnostics()
        target_diag = self._advantage_target_summary()

        diagnostics["iteration"].append(int(it + 1))
        diagnostics["wall_clock_seconds"].append(float(time.perf_counter() - start_time))
        diagnostics["strategy_buffer_size"].append(int(len(self._strategy_memories)))
        diagnostics["advantage_buffer_size_player_0"].append(int(len(self._advantage_memories[0])))
        diagnostics["advantage_buffer_size_player_1"].append(int(len(self._advantage_memories[1])))
        diagnostics["policy_loss"].append(np.nan if pol_loss is None else float(pol_loss))
        diagnostics["advantage_grad_norm_player_0"].append(float(self._last_advantage_grad_norm[0]))
        diagnostics["advantage_grad_norm_player_1"].append(float(self._last_advantage_grad_norm[1]))
        diagnostics["policy_grad_norm"].append(float(self._last_policy_grad_norm))
        diagnostics["legal_action_mass_mean"].append(float(policy_diag["legal_action_mass_mean"]))
        diagnostics["legal_action_mass_min"].append(float(policy_diag["legal_action_mass_min"]))
        diagnostics["policy_entropy_mean"].append(float(policy_diag["entropy_mean"]))
        diagnostics["policy_normalized_entropy_mean"].append(float(policy_diag["normalized_entropy_mean"]))
        diagnostics["advantage_target_count"].append(int(target_diag["count"]))
        diagnostics["advantage_target_mean"].append(float(target_diag["mean"]))
        diagnostics["advantage_target_variance"].append(float(target_diag["variance"]))
        diagnostics["advantage_target_abs_mean"].append(float(target_diag["abs_mean"]))

    self._nodes_touched_history = nodes_touched_history
    self._average_policy_value_history = average_policy_values
    return (self._policy_network, advantage_losses, policy_losses_at_checkpoints,
            convs, nodes_touched_history, average_policy_values, dict(diagnostics))

  def _traverse_game_tree(self, state, player):
    """Performs a traversal of the game tree.

    Over a traversal the advantage and strategy memories are populated with
    computed advantage values and matched regrets respectively.

    Args:
      state: Current OpenSpiel game state.
      player: (int) Player index for this traversal.

    Returns:
      (float) Recursively returns expected payoffs for each action.
    """
    self._nodes_touched += 1
    expected_payoff = collections.defaultdict(float)
    if state.is_terminal():
      # Terminal state get returns.
      return state.returns()[player]
    elif state.is_chance_node():
      # If this is a chance node, sample an action
      chance_outcome, chance_proba = zip(*state.chance_outcomes())
      action = np.random.choice(chance_outcome, p=chance_proba)
      return self._traverse_game_tree(state.child(action), player)
    elif state.current_player() == player:
      sampled_regret = collections.defaultdict(float)
      # Update the policy over the info set & actions via regret matching.
      _, strategy = self._sample_action_from_advantage(state, player)
      for action in state.legal_actions():
        expected_payoff[action] = self._traverse_game_tree(
            state.child(action), player)
      cfv = 0
      for a_ in state.legal_actions():
        cfv += strategy[a_] * expected_payoff[a_]
      for action in state.legal_actions():
        sampled_regret[action] = expected_payoff[action]
        sampled_regret[action] -= cfv
      sampled_regret_arr = [0] * self._num_actions
      for action in sampled_regret:
        sampled_regret_arr[action] = sampled_regret[action]
      self._advantage_memories[player].add(
          AdvantageMemory(state.information_state_tensor(), int(self._iteration),
                          sampled_regret_arr, action))
      return cfv
    else:
      other_player = state.current_player()
      _, strategy = self._sample_action_from_advantage(state, other_player)
      # Recompute distribution for numerical errors.
      probs = np.array(strategy)
      probs /= probs.sum()
      sampled_action = np.random.choice(range(self._num_actions), p=probs)
      self._strategy_memories.add(
          StrategyMemory(
              state.information_state_tensor(other_player), int(self._iteration),
              strategy))
      return self._traverse_game_tree(state.child(sampled_action), player)

  def _sample_action_from_advantage(self, state, player):
    """Returns an info state policy by applying regret-matching.

    Args:
      state: Current OpenSpiel game state.
      player: (int) Player index over which to compute regrets.

    Returns:
      1. (list) Advantage values for info state actions indexed by action.
      2. (list) Matched regrets, prob for actions indexed by action.
    """
    info_state = state.information_state_tensor(player)
    legal_actions = state.legal_actions(player)
    with torch.no_grad():
      state_tensor = torch.FloatTensor(np.expand_dims(info_state, axis=0))
      raw_advantages = self._advantage_networks[player](state_tensor)[0].numpy()
    advantages = [max(0., advantage) for advantage in raw_advantages]
    cumulative_regret = np.sum([advantages[action] for action in legal_actions])
    matched_regrets = np.array([0.] * self._num_actions)
    if cumulative_regret > 0.:
      for action in legal_actions:
        matched_regrets[action] = advantages[action] / cumulative_regret
    else:
      # Standard regret matching falls back to a uniform strategy when all
      # positive regrets are zero. This avoids deterministic tie-breaking that
      # can artificially reduce exploration.
      for action in legal_actions:
        matched_regrets[action] = 1.0 / len(legal_actions)
    return advantages, matched_regrets

  def action_probabilities(self, state, player_id=None):
    """Computes legal-action probabilities for the current player in state.

    The network produces logits for the full action space. Illegal actions are
    masked out and the probability mass over legal actions is renormalised.
    This is important for OpenSpiel evaluation routines, which expect the
    returned legal-action distribution to sum to one.
    """
    cur_player = state.current_player() if player_id is None else player_id
    legal_actions = state.legal_actions(cur_player)
    if not legal_actions:
      return {}
    info_state_vector = np.asarray(state.information_state_tensor(cur_player), dtype=np.float32)
    if len(info_state_vector.shape) == 1:
      info_state_vector = np.expand_dims(info_state_vector, axis=0)
    with torch.no_grad():
      logits = self._policy_network(torch.FloatTensor(info_state_vector))[0]
      legal_logits = logits[legal_actions]
      legal_probs = torch.softmax(legal_logits, dim=-1).cpu().numpy()
    return {action: float(prob) for action, prob in zip(legal_actions, legal_probs)}

  @staticmethod
  def _gradient_norm(parameters):
    """Returns the global L2 norm of gradients for a parameter iterable."""
    total_sq = 0.0
    for parameter in parameters:
      if parameter.grad is not None:
        grad = parameter.grad.detach()
        total_sq += float(torch.sum(grad * grad).cpu().item())
    return math.sqrt(total_sq)

  def _policy_network_diagnostics(self):
    """Summarises legal-action mass and entropy over all reachable infosets."""
    stats = []
    seen = set()

    def visit(state):
      if state.is_terminal():
        return
      if state.is_chance_node():
        for action, _ in state.chance_outcomes():
          visit(state.child(action))
        return

      player = state.current_player()
      legal_actions = state.legal_actions(player)
      info_state = np.asarray(state.information_state_tensor(player), dtype=np.float32)
      key = (player, tuple(info_state.tolist()))
      if key not in seen and legal_actions:
        seen.add(key)
        with torch.no_grad():
          logits = self._policy_network(
              torch.FloatTensor(np.expand_dims(info_state, axis=0)))[0]
          full_probs = self._policy_sm(logits).cpu().numpy()
        legal_mass = float(np.sum(full_probs[legal_actions]))
        if legal_mass > 0:
          legal_probs = full_probs[legal_actions] / legal_mass
        else:
          legal_probs = np.ones(len(legal_actions), dtype=np.float32) / len(legal_actions)
        entropy = float(-np.sum(legal_probs * np.log(legal_probs + 1e-12)))
        max_entropy = float(np.log(len(legal_actions))) if len(legal_actions) > 1 else 0.0
        norm_entropy = entropy / max_entropy if max_entropy > 0 else 0.0
        stats.append((legal_mass, entropy, norm_entropy))

      for action in legal_actions:
        visit(state.child(action))

    visit(self._game.new_initial_state())
    if not stats:
      return {
          "legal_action_mass_mean": np.nan,
          "legal_action_mass_min": np.nan,
          "entropy_mean": np.nan,
          "normalized_entropy_mean": np.nan,
      }
    arr = np.asarray(stats, dtype=np.float32)
    return {
        "legal_action_mass_mean": float(np.mean(arr[:, 0])),
        "legal_action_mass_min": float(np.min(arr[:, 0])),
        "entropy_mean": float(np.mean(arr[:, 1])),
        "normalized_entropy_mean": float(np.mean(arr[:, 2])),
    }

  def _advantage_target_summary(self):
    """Summarises sampled advantage targets in the replay buffers."""
    values = []
    for buffer in self._advantage_memories:
      for sample in buffer:
        values.extend(np.asarray(sample.advantage, dtype=np.float32).reshape(-1).tolist())
    if not values:
      return {"count": 0, "mean": np.nan, "variance": np.nan, "abs_mean": np.nan}
    values = np.asarray(values, dtype=np.float32)
    return {
        "count": int(values.size),
        "mean": float(np.mean(values)),
        "variance": float(np.var(values)),
        "abs_mean": float(np.mean(np.abs(values))),
    }

  def _learn_advantage_network(self, player):
    """Compute the loss on sampled transitions and update an advantage network."""
    last_loss = None
    grad_norms = []
    for _ in range(self._advantage_network_train_steps):
      if self._batch_size_advantage:
        if self._batch_size_advantage > len(self._advantage_memories[player]):
          return None
        samples = self._advantage_memories[player].sample(self._batch_size_advantage)
      else:
        samples = self._advantage_memories[player]

      info_states = []
      advantages = []
      iterations = []
      for s in samples:
        info_states.append(s.info_state)
        advantages.append(s.advantage)
        iterations.append(self._as_scalar_iteration(s.iteration))
      if not info_states:
        return None

      self._optimizer_advantages[player].zero_grad()
      advantages = torch.FloatTensor(np.asarray(advantages, dtype=np.float32))
      iter_weights = np.sqrt(np.asarray(iterations, dtype=np.float32)).reshape(-1, 1)
      iters = torch.FloatTensor(iter_weights)
      outputs = self._advantage_networks[player](
          torch.FloatTensor(np.asarray(info_states, dtype=np.float32)))
      loss_advantages = self._loss_advantages(iters * outputs, iters * advantages)
      loss_advantages.backward()
      grad_norms.append(self._gradient_norm(self._advantage_networks[player].parameters()))
      self._optimizer_advantages[player].step()
      last_loss = float(loss_advantages.detach().cpu().item())

    self._last_advantage_grad_norm[player] = float(np.mean(grad_norms)) if grad_norms else np.nan
    return last_loss

  def _learn_strategy_network(self):
    """Compute the loss over the average-policy network."""
    last_loss = None
    grad_norms = []
    for _ in range(self._policy_network_train_steps):
      if self._batch_size_strategy:
        if self._batch_size_strategy > len(self._strategy_memories):
          return None
        samples = self._strategy_memories.sample(self._batch_size_strategy)
      else:
        samples = self._strategy_memories

      info_states = []
      action_probs = []
      iterations = []
      for s in samples:
        info_states.append(s.info_state)
        action_probs.append(s.strategy_action_probs)
        iterations.append(self._as_scalar_iteration(s.iteration))
      if not info_states:
        return None

      self._optimizer_policy.zero_grad()
      iter_weights = np.sqrt(np.asarray(iterations, dtype=np.float32)).reshape(-1, 1)
      iters = torch.FloatTensor(iter_weights)
      ac_probs = torch.FloatTensor(np.asarray(action_probs, dtype=np.float32))
      logits = self._policy_network(torch.FloatTensor(np.asarray(info_states, dtype=np.float32)))
      outputs = self._policy_sm(logits)
      loss_strategy = self._loss_policy(iters * outputs, iters * ac_probs)
      loss_strategy.backward()
      grad_norms.append(self._gradient_norm(self._policy_network.parameters()))
      self._optimizer_policy.step()
      last_loss = float(loss_strategy.detach().cpu().item())

    self._last_policy_grad_norm = float(np.mean(grad_norms)) if grad_norms else np.nan
    return last_loss
  
  def extract_full_model(self, include_buffers=True, include_rng_state=True):
    """Extracts a full Deep CFR checkpoint as a Python dict.

    The returned dict is designed to be serialized with `torch.save(...)` and
    later restored with `load_full_model(...)`.

    Args:
      include_buffers: Whether to include the reservoir replay buffers (needed
        to resume training properly).
      include_rng_state: Whether to include Python/NumPy/PyTorch RNG state.

    Returns:
      A checkpoint dict.
    """
    ckpt = {
        "version": 1,
        "iteration": int(getattr(self, "_iteration", 0)),

        # Models + optimizers
        "policy_state_dict": self._policy_network.state_dict(),
        "policy_opt_state_dict": self._optimizer_policy.state_dict(),
        "advantage_state_dicts": [net.state_dict() for net in self._advantage_networks],
        "advantage_opt_state_dicts": [opt.state_dict() for opt in self._optimizer_advantages],

        # Lightweight sanity metadata
        "meta": {
            "num_players": int(self._num_players),
            "num_actions": int(self._num_actions),
            "embedding_size": int(self._embedding_size),
        },
    }

    if include_buffers:
      ckpt["strategy_buffer"] = {
          "capacity": int(self._strategy_memories._reservoir_buffer_capacity),
          "add_calls": int(self._strategy_memories._add_calls),
          "data": list(self._strategy_memories._data),
      }
      ckpt["advantage_buffers"] = [
          {
              "capacity": int(buf._reservoir_buffer_capacity),
              "add_calls": int(buf._add_calls),
              "data": list(buf._data),
          }
          for buf in self._advantage_memories
      ]

    if include_rng_state:
      ckpt["rng_state"] = {
          "python": random.getstate(),
          "numpy": np.random.get_state(),
          "torch": torch.get_rng_state(),
      }
      if torch.cuda.is_available():
        try:
          ckpt["rng_state"]["torch_cuda"] = torch.cuda.get_rng_state_all()
        except Exception:
          pass

    return ckpt

  def load_full_model(self,
                      ckpt_or_path,
                      map_location=None,
                      strict=True,
                      restore_buffers=True,
                      restore_rng_state=True):
    """Restores a full Deep CFR checkpoint (dict or path) into this solver.

    Args:
      ckpt_or_path: A checkpoint dict produced by `extract_full_model()` or a
        filesystem path to a `torch.save`'d checkpoint.
      map_location: Passed to `torch.load` when `ckpt_or_path` is a path.
        Use "cpu" to load GPU checkpoints on a CPU-only machine.
      strict: Passed to `nn.Module.load_state_dict(...)` for the networks.
      restore_buffers: Whether to restore replay buffers.
      restore_rng_state: Whether to restore Python/NumPy/PyTorch RNG state.

    Returns:
      self
    """

    def _move_optimizer_state_to_device(optimizer, device):
      # Optimizer state tensors may be on the wrong device after torch.load.
      for state in optimizer.state.values():
        for k, v in state.items():
          if torch.is_tensor(v):
            state[k] = v.to(device)

    if isinstance(ckpt_or_path, (str, os.PathLike)):
      ckpt = torch.load(ckpt_or_path, map_location=map_location, weights_only=False)
    else:
      ckpt = ckpt_or_path

    # Restore iteration counter (used inside losses)
    self._iteration = int(ckpt.get("iteration", 0))

    # Restore models + optimizers
    self._policy_network.load_state_dict(ckpt["policy_state_dict"], strict=strict)
    self._optimizer_policy.load_state_dict(ckpt["policy_opt_state_dict"])
    _move_optimizer_state_to_device(self._optimizer_policy,
                                   next(self._policy_network.parameters()).device)

    for net, sd in zip(self._advantage_networks, ckpt["advantage_state_dicts"]):
      net.load_state_dict(sd, strict=strict)
    for opt, net, sd in zip(self._optimizer_advantages,
                            self._advantage_networks,
                            ckpt["advantage_opt_state_dicts"]):
      opt.load_state_dict(sd)
      _move_optimizer_state_to_device(opt, next(net.parameters()).device)

    # Restore buffers
    if restore_buffers:
      if "strategy_buffer" in ckpt:
        sb = ckpt["strategy_buffer"]
        self._strategy_memories._reservoir_buffer_capacity = int(sb["capacity"])
        self._strategy_memories._add_calls = int(sb["add_calls"])
        self._strategy_memories._data = list(sb["data"])
      if "advantage_buffers" in ckpt:
        for buf, saved in zip(self._advantage_memories, ckpt["advantage_buffers"]):
          buf._reservoir_buffer_capacity = int(saved["capacity"])
          buf._add_calls = int(saved["add_calls"])
          buf._data = list(saved["data"])

    # Optional: restore RNG state
    if restore_rng_state:
      rng = ckpt.get("rng_state", {})
      if "python" in rng: random.setstate(rng["python"])
      if "numpy" in rng: np.random.set_state(rng["numpy"])
      if "torch" in rng: torch.set_rng_state(rng["torch"])
      if torch.cuda.is_available() and "torch_cuda" in rng:
        try:
          torch.cuda.set_rng_state_all(rng["torch_cuda"])
        except Exception:
          pass

    return self

In [ ]:
def set_seed(seed):
  """Sets Python, NumPy, and PyTorch RNG seeds for reproducibility."""
  seed = int(seed)
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False


KUHN_GAME_VALUE_PLAYER_0 = -1.0 / 18.0
EXPLOITABILITY_THRESHOLD = 0.05


## Experimental configuration

The protocol uses 1,500 iterations and evaluates the average policy every 25 iterations. This is deliberately a single fixed-budget run per seed; checkpoint reloading has been removed from the experimental protocol.

In [ ]:
EXPERIMENT_CONFIG = {
    "experiment_name": "kuhn_poker_deep_cfr_multiseed_validation",
    "game_name": "kuhn_poker",
    "num_iterations": 1500,
    "num_traversals": 320,
    "evaluation_interval": 25,
    "policy_network_layers": (32, 32),
    "advantage_network_layers": (32, 32),
    "learning_rate": 0.003,
    "batch_size_advantage": 1024,
    "batch_size_strategy": 1024,
    "memory_capacity": int(1e7),
    "reinitialize_advantage_networks": False,
    "policy_network_train_steps": 200,
    "advantage_network_train_steps": 200,
    "compute_exploitability": True,
    "exploitability_threshold": EXPLOITABILITY_THRESHOLD,
}

# Ten fixed seeds make the result reproducible and allow mean/uncertainty reporting.
SEEDS = [1234, 2025, 31415, 27182, 16180, 4242, 8675309, 7, 99, 1001]

SAVE_FINAL_CHECKPOINTS = False
EXPORT_ROOT = Path("chart_exports")


In [ ]:
def make_solver(game, config):
  return DeepCFRSolver(
      game,
      policy_network_layers=config["policy_network_layers"],
      advantage_network_layers=config["advantage_network_layers"],
      num_iterations=config["num_iterations"],
      num_traversals=config["num_traversals"],
      learning_rate=config["learning_rate"],
      batch_size_advantage=config["batch_size_advantage"],
      batch_size_strategy=config["batch_size_strategy"],
      memory_capacity=config["memory_capacity"],
      reinitialize_advantage_networks=config["reinitialize_advantage_networks"],
      policy_network_train_steps=config["policy_network_train_steps"],
      advantage_network_train_steps=config["advantage_network_train_steps"],
      compute_exploitability=config["compute_exploitability"],
      policy_network_train_every=config["evaluation_interval"],
  )


def first_nodes_to_threshold(nodes, metric, threshold):
  idx = np.where(np.asarray(metric) <= threshold)[0]
  return np.nan if len(idx) == 0 else float(np.asarray(nodes)[idx[0]])


def first_time_to_threshold(times, metric, threshold):
  idx = np.where(np.asarray(metric) <= threshold)[0]
  return np.nan if len(idx) == 0 else float(np.asarray(times)[idx[0]])


def final_window_mean(values, window=5):
  values = np.asarray(values, dtype=np.float64)
  if values.size == 0:
    return np.nan
  return float(np.mean(values[-min(window, values.size):]))


def run_single_seed(seed, config, export_dir=None):
  set_seed(seed)
  game = pyspiel.load_game(config["game_name"])
  solver = make_solver(game, config)

  (policy_network, advantage_losses, policy_losses, convs, nodes_touched,
   avg_policy_values, diagnostics) = solver.solve()

  exploitability_curve = np.asarray(convs, dtype=np.float64) / 2.0
  nodes_touched = np.asarray(nodes_touched, dtype=np.float64)
  avg_policy_values = np.asarray(avg_policy_values, dtype=np.float64)
  value_error = np.abs(avg_policy_values - KUHN_GAME_VALUE_PLAYER_0)

  diagnostics = {k: np.asarray(v) for k, v in diagnostics.items()}
  iterations = diagnostics["iteration"].astype(int)
  wall_clock = diagnostics["wall_clock_seconds"].astype(float)

  final_policy = policy.tabular_policy_from_callable(game, solver.action_probabilities)
  final_nash_conv = exploitability.nash_conv(game, final_policy)
  final_policy_value = expected_game_score.policy_value(
      game.new_initial_state(), [final_policy] * game.num_players())[0]

  summary = {
      "seed": int(seed),
      "final_exploitability": float(exploitability_curve[-1]),
      "best_exploitability": float(np.min(exploitability_curve)),
      "final_window_mean_exploitability": final_window_mean(exploitability_curve),
      "final_policy_value": float(final_policy_value),
      "final_policy_value_error": float(abs(final_policy_value - KUHN_GAME_VALUE_PLAYER_0)),
      "best_policy_value_error": float(np.min(value_error)),
      "final_nodes_touched": float(nodes_touched[-1]),
      "final_wall_clock_seconds": float(wall_clock[-1]),
      "nodes_to_exploitability_threshold": first_nodes_to_threshold(
          nodes_touched, exploitability_curve, config["exploitability_threshold"]),
      "seconds_to_exploitability_threshold": first_time_to_threshold(
          wall_clock, exploitability_curve, config["exploitability_threshold"]),
      "final_legal_action_mass_mean": float(diagnostics["legal_action_mass_mean"][-1]),
      "final_legal_action_mass_min": float(diagnostics["legal_action_mass_min"][-1]),
      "final_policy_normalized_entropy_mean": float(diagnostics["policy_normalized_entropy_mean"][-1]),
      "final_advantage_target_variance": float(diagnostics["advantage_target_variance"][-1]),
      "final_policy_loss": float(diagnostics["policy_loss"][-1]),
      "final_policy_grad_norm": float(diagnostics["policy_grad_norm"][-1]),
      "final_advantage_grad_norm_player_0": float(diagnostics["advantage_grad_norm_player_0"][-1]),
      "final_advantage_grad_norm_player_1": float(diagnostics["advantage_grad_norm_player_1"][-1]),
      "final_nash_conv_recomputed": float(final_nash_conv),
  }

  result = {
      "seed": int(seed),
      "iterations": iterations,
      "nodes_touched": nodes_touched,
      "wall_clock_seconds": wall_clock,
      "exploitability": exploitability_curve,
      "average_policy_value": avg_policy_values,
      "policy_value_error": value_error,
      "diagnostics": diagnostics,
      "summary": summary,
  }

  if SAVE_FINAL_CHECKPOINTS and export_dir is not None:
    checkpoint_dir = Path(export_dir) / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    torch.save(solver.extract_full_model(), checkpoint_dir / f"seed_{seed}_final_model.pt")

  return result


In [ ]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = EXPORT_ROOT / f"{EXPERIMENT_CONFIG['experiment_name']}_{run_id}"
run_dir.mkdir(parents=True, exist_ok=True)

print("Export directory:", run_dir.resolve())
print("Running seeds:", SEEDS)

results = []
for seed in tqdm(SEEDS, desc="Deep CFR seeds"):
  results.append(run_single_seed(seed, EXPERIMENT_CONFIG, export_dir=run_dir))

print("Completed", len(results), "seeds")


In [ ]:
summary_rows = [result["summary"] for result in results]
summary_fields = list(summary_rows[0].keys())

summary_csv = run_dir / "seed_summary.csv"
with open(summary_csv, "w", newline="", encoding="utf-8") as f:
  writer = csv.DictWriter(f, fieldnames=summary_fields)
  writer.writeheader()
  writer.writerows(summary_rows)

for key, value in EXPERIMENT_CONFIG.items():
  if isinstance(value, tuple):
    EXPERIMENT_CONFIG[key] = list(value)
metadata = {
    "experiment_config": EXPERIMENT_CONFIG,
    "seeds": SEEDS,
    "kuhn_game_value_player_0": KUHN_GAME_VALUE_PLAYER_0,
    "torch_version": torch.__version__,
    "pyspiel_version": getattr(pyspiel, "__version__", "unknown"),
}
with open(run_dir / "experiment_metadata.json", "w", encoding="utf-8") as f:
  json.dump(metadata, f, indent=2)

# Print a compact summary for the notebook.
print("\nPer-seed summary saved to:", summary_csv.resolve())
print("\nFinal exploitability by seed:")
for row in summary_rows:
  print(f"  seed={row['seed']}: final={row['final_exploitability']:.6f}, "
        f"best={row['best_exploitability']:.6f}, "
        f"final value error={row['final_policy_value_error']:.6f}")

summary_numeric = {}
for field in summary_fields:
  if field == "seed":
    continue
  vals = np.asarray([row[field] for row in summary_rows], dtype=np.float64)
  finite = vals[np.isfinite(vals)]
  if finite.size:
    summary_numeric[field] = {
        "mean": float(np.mean(finite)),
        "std": float(np.std(finite, ddof=1)) if finite.size > 1 else 0.0,
        "se": float(stats.sem(finite)) if finite.size > 1 else 0.0,
        "n_finite": int(finite.size),
    }

with open(run_dir / "aggregate_summary.json", "w", encoding="utf-8") as f:
  json.dump(summary_numeric, f, indent=2)

print("\nAggregate final exploitability:", summary_numeric["final_exploitability"])
print("Aggregate best exploitability:", summary_numeric["best_exploitability"])
print("Aggregate final policy-value error:", summary_numeric["final_policy_value_error"])


In [ ]:
def stack_curve(results, key):
  return np.vstack([np.asarray(result[key], dtype=np.float64) for result in results])

iterations = results[0]["iterations"]
exploitability_mat = stack_curve(results, "exploitability")
value_error_mat = stack_curve(results, "policy_value_error")
nodes_mat = stack_curve(results, "nodes_touched")
wall_clock_mat = stack_curve(results, "wall_clock_seconds")

mean_exploitability = np.mean(exploitability_mat, axis=0)
se_exploitability = stats.sem(exploitability_mat, axis=0)
mean_value_error = np.mean(value_error_mat, axis=0)
se_value_error = stats.sem(value_error_mat, axis=0)
mean_nodes = np.mean(nodes_mat, axis=0)
mean_wall_clock = np.mean(wall_clock_mat, axis=0)

# Plot exploitability against training iteration.
fig, ax = plt.subplots(figsize=(8, 5))
for result in results:
  ax.plot(result["iterations"], result["exploitability"], alpha=0.25, linewidth=1)
ax.plot(iterations, mean_exploitability, linewidth=2, label="Mean across seeds")
ax.fill_between(iterations, mean_exploitability - se_exploitability,
                mean_exploitability + se_exploitability, alpha=0.2,
                label="Mean $\\pm$ s.e.")
ax.axhline(EXPLOITABILITY_THRESHOLD, linestyle="--", label="Exploitability threshold")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Exploitability (NashConv/2)")
ax.set_title("Kuhn Poker Deep CFR: Exploitability Across Seeds")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / "exploitability_by_iteration_multiseed.png", dpi=200, bbox_inches="tight")
plt.show()

# Plot exploitability against nodes touched.
fig, ax = plt.subplots(figsize=(8, 5))
for result in results:
  ax.plot(result["nodes_touched"], result["exploitability"], alpha=0.25, linewidth=1)
ax.plot(mean_nodes, mean_exploitability, linewidth=2, label="Mean across seeds")
ax.fill_between(mean_nodes, mean_exploitability - se_exploitability,
                mean_exploitability + se_exploitability, alpha=0.2,
                label="Mean $\\pm$ s.e.")
ax.axhline(EXPLOITABILITY_THRESHOLD, linestyle="--", label="Exploitability threshold")
ax.set_xlabel("Nodes touched")
ax.set_ylabel("Exploitability (NashConv/2)")
ax.set_title("Kuhn Poker Deep CFR: Exploitability by Nodes Touched")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / "exploitability_by_nodes_multiseed.png", dpi=200, bbox_inches="tight")
plt.show()

# Plot policy-value error from the known Kuhn value.
fig, ax = plt.subplots(figsize=(8, 5))
for result in results:
  ax.plot(result["iterations"], result["policy_value_error"], alpha=0.25, linewidth=1)
ax.plot(iterations, mean_value_error, linewidth=2, label="Mean across seeds")
ax.fill_between(iterations, mean_value_error - se_value_error,
                mean_value_error + se_value_error, alpha=0.2,
                label="Mean $\\pm$ s.e.")
ax.set_xlabel("Training iteration")
ax.set_ylabel(r"$|v(\\sigma) - (-1/18)|$")
ax.set_title("Kuhn Poker Deep CFR: Policy-Value Error")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / "policy_value_error_multiseed.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Diagnostic plots: these are not primary strategic performance metrics.
policy_loss_mat = np.vstack([result["diagnostics"]["policy_loss"].astype(float) for result in results])
adv_target_var_mat = np.vstack([result["diagnostics"]["advantage_target_variance"].astype(float) for result in results])
entropy_mat = np.vstack([result["diagnostics"]["policy_normalized_entropy_mean"].astype(float) for result in results])
legal_mass_mat = np.vstack([result["diagnostics"]["legal_action_mass_mean"].astype(float) for result in results])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iterations, np.mean(policy_loss_mat, axis=0), linewidth=2, label="Policy loss")
ax.fill_between(iterations,
                np.mean(policy_loss_mat, axis=0) - stats.sem(policy_loss_mat, axis=0),
                np.mean(policy_loss_mat, axis=0) + stats.sem(policy_loss_mat, axis=0),
                alpha=0.2)
ax.set_xlabel("Training iteration")
ax.set_ylabel("MSE loss")
ax.set_title("Average-Policy Network Loss Diagnostic")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / "policy_loss_diagnostic.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iterations, np.mean(adv_target_var_mat, axis=0), linewidth=2)
ax.set_xlabel("Training iteration")
ax.set_ylabel("Advantage target variance")
ax.set_title("Advantage-Target Variance Diagnostic")
ax.grid(True)
fig.tight_layout()
fig.savefig(run_dir / "advantage_target_variance_diagnostic.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iterations, np.mean(entropy_mat, axis=0), linewidth=2, label="Normalised entropy")
ax.plot(iterations, np.mean(legal_mass_mat, axis=0), linewidth=2, label="Raw legal-action mass before masking")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Diagnostic value")
ax.set_title("Policy Distribution Diagnostics")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir / "policy_distribution_diagnostics.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Save full per-checkpoint curves in CSV form for thesis plots and later analysis.
curve_csv = run_dir / "checkpoint_curves.csv"
curve_fields = [
    "seed", "iteration", "nodes_touched", "wall_clock_seconds", "exploitability",
    "average_policy_value", "policy_value_error", "policy_loss",
    "strategy_buffer_size", "advantage_buffer_size_player_0", "advantage_buffer_size_player_1",
    "legal_action_mass_mean", "legal_action_mass_min", "policy_normalized_entropy_mean",
    "advantage_target_variance", "policy_grad_norm",
    "advantage_grad_norm_player_0", "advantage_grad_norm_player_1",
]
with open(curve_csv, "w", newline="", encoding="utf-8") as f:
  writer = csv.DictWriter(f, fieldnames=curve_fields)
  writer.writeheader()
  for result in results:
    diag = result["diagnostics"]
    for i, iteration in enumerate(result["iterations"]):
      writer.writerow({
          "seed": result["seed"],
          "iteration": int(iteration),
          "nodes_touched": float(result["nodes_touched"][i]),
          "wall_clock_seconds": float(result["wall_clock_seconds"][i]),
          "exploitability": float(result["exploitability"][i]),
          "average_policy_value": float(result["average_policy_value"][i]),
          "policy_value_error": float(result["policy_value_error"][i]),
          "policy_loss": float(diag["policy_loss"][i]),
          "strategy_buffer_size": int(diag["strategy_buffer_size"][i]),
          "advantage_buffer_size_player_0": int(diag["advantage_buffer_size_player_0"][i]),
          "advantage_buffer_size_player_1": int(diag["advantage_buffer_size_player_1"][i]),
          "legal_action_mass_mean": float(diag["legal_action_mass_mean"][i]),
          "legal_action_mass_min": float(diag["legal_action_mass_min"][i]),
          "policy_normalized_entropy_mean": float(diag["policy_normalized_entropy_mean"][i]),
          "advantage_target_variance": float(diag["advantage_target_variance"][i]),
          "policy_grad_norm": float(diag["policy_grad_norm"][i]),
          "advantage_grad_norm_player_0": float(diag["advantage_grad_norm_player_0"][i]),
          "advantage_grad_norm_player_1": float(diag["advantage_grad_norm_player_1"][i]),
      })

np.savez_compressed(
    run_dir / "multiseed_curves.npz",
    seeds=np.asarray(SEEDS),
    iterations=iterations,
    exploitability=exploitability_mat,
    policy_value_error=value_error_mat,
    average_policy_value=np.vstack([result["average_policy_value"] for result in results]),
    nodes_touched=nodes_mat,
    wall_clock_seconds=wall_clock_mat,
    mean_exploitability=mean_exploitability,
    se_exploitability=se_exploitability,
    mean_policy_value_error=mean_value_error,
    se_policy_value_error=se_value_error,
)

print("Saved checkpoint curves to:", curve_csv.resolve())
print("Saved all outputs to:", run_dir.resolve())
